# **Análise baseada especifica baseada na região caririense**

> Totais de Cidades inclusas na análise: 28

Abaiara, Altaneira, Antonina do Norte, Araripe, Assaré, Aurora, Barbalha, Barro, Brejo Santo, Campos Sales, Caririaçu, Crato, Farias Brito, Granjeiro, Jardim, Jati, Juazeiro do Norte, Mauriti, Milagres, Missão Velha, Nova Olinda, Penaforte, Porteiras, Potengi, Salitre, Santana do Cariri, Tarrafas, Várzea Alegre.

#### _Importando as dependencias_

In [50]:
import pandas as pd                          # Manipulação dos dados
import numpy as np                           # Calculos Matmáticos
import matplotlib.pyplot as plt              # Geração de Gráficos
import seaborn as sns                        # Graficos automaticos, baseado no matplotlib
import statistics as sta                     # Biblioteca padrão do py para estatistica
from scipy import stats                      # Estastistica avançada
import os                                    # SO

### **Limpeza e Preparação**

In [51]:
cito_mamo = pd.read_csv('../../Datasets/cito_e_mamo_cariri.csv')
print(cito_mamo.columns.tolist())
cito_mamo.head(10)

['estado', 'municipio', 'valor_indicador', 'ano_mes', 'data_atualizacao', 'tipo_exame']


,estado,municipio,valor_indicador,ano_mes,data_atualizacao,tipo_exame
0,CE,Brejo Santo,748.0,201612,2026-03-17 01:25:03,Citopatologico
1,CE,Crato,16742.0,201612,2026-03-17 01:25:03,Citopatologico
2,CE,Juazeiro do Norte,4176.0,201612,2026-03-17 01:25:03,Citopatologico
3,CE,Barbalha,1.0,201712,2026-03-17 01:22:16,Citopatologico
4,CE,Brejo Santo,4251.0,201712,2026-03-17 01:22:16,Citopatologico
5,CE,Crato,19052.0,201712,2026-03-17 01:22:16,Citopatologico
6,CE,Juazeiro do Norte,4649.0,201712,2026-03-17 01:22:16,Citopatologico
7,CE,Barbalha,33.0,201812,2026-03-17 01:20:07,Citopatologico
8,CE,Brejo Santo,5414.0,201812,2026-03-17 01:20:07,Citopatologico
9,CE,Crato,13976.0,201812,2026-03-17 01:20:07,Citopatologico


#### _Visão geral do dataset cito_e_mamo.csv_

Observação e primeira visão dos dados e tipos de dados:
- Tipo de valor, valores nulos, valores unicos (quantidade de valores diferentes)... 
-  Verificação das variáveis disponíveis e suas escalas (nominal, ordinal, discreta, contínua, intervalar, etc.).

Estastica: 
- Média, Mediana, Mínimo, Máximo e Desvio padrão para a coluna do valor_indicador

> Foi utilizado a métrica IQR por não ser sensível a Outlier como a métrica tradicional


##### _Classificação das variáveis por escala de medida_

Aqui pegamos o tipo, escala e valores unicos e de cada coluna e criamos uma tabela para as informações

In [56]:
def infer_scale(column_series):
    if pd.api.types.is_datetime64_any_dtype(column_series):
        return 'Temporal (intervalar/razão)'

    if column_series.dtype == 'string' or column_series.dtype == object:
        try:
            converted = pd.to_datetime(column_series, errors='coerce')
            if converted.notna().sum() / len(column_series) > 0.9:
                return 'Temporal (intervalar/razão) - string'
        except Exception:
            pass
        return 'Nominal'  # strings são normalmente nominais

    if pd.api.types.is_categorical_dtype(column_series):
        if getattr(column_series.dtype, 'ordered', False):
            return 'Ordinal'
        return 'Nominal'

    if pd.api.types.is_integer_dtype(column_series):
        unique = column_series.nunique(dropna=True)
        if unique < 30:
            return 'Discreta (categoria numérica/ordinal)'
        return 'Discreta'

    if pd.api.types.is_float_dtype(column_series):
        return 'Contínua'

    return 'Desconhecida'

scale_info = []
for col in cito_mamo.columns:
    dtype = cito_mamo[col].dtype
    scale = infer_scale(cito_mamo[col])
    scale_info.append({'coluna': col, 'tipo': str(dtype), 'escala': scale, 'valores_unicos': cito_mamo[col].nunique(dropna=True)})

scale_df = pd.DataFrame(scale_info)
scale_df


C:\Users\tecla\AppData\Local\Temp\ipykernel_23184\2486103854.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(column_series, errors='coerce')
C:\Users\tecla\AppData\Local\Temp\ipykernel_23184\2486103854.py:14: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(column_series):
C:\Users\tecla\AppData\Local\Temp\ipykernel_23184\2486103854.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(column_series, errors='coerce')


,coluna,tipo,escala,valores_unicos
0,estado,str,Nominal,1
1,municipio,str,Nominal,17
2,valor_indicador,float64,Contínua,108
3,ano_mes,int64,Discreta (categoria numérica/ordinal),10
4,data_atualizacao,str,Temporal (intervalar/razão) - string,20
5,tipo_exame,str,Nominal,2


##### _Entendimento mais detalhado_

In [ ]:
print("# DOCUMENTAÇÃO DO DATASET - EXAMES CITOPATOLÓGICOS\n")
print("## Visão Geral")
print(f"- **Total de registros**: {len(cito_mamo):,}")
print(f"- **Total de colunas**: {len(cito_mamo.columns)}")
print(f"- **Última atualização**: 24 de novembro de 2025\n")

print("## Descrição Detalhada das Colunas\n")

for i, column in enumerate(cito_mamo.columns, 1):
    print(f"### {i}. {column}")
    print(f"**Tipo de dado**: {cito_mamo[column].dtype}")
    print(f"**Valores únicos**: {cito_mamo[column].nunique():,}")
    print(f"**Valores nulos**: {cito_mamo[column].isnull().sum():,} ({cito_mamo[column].isnull().sum()/len(cito_mamo)*100:.1f}%)")

    if cito_mamo[column].dtype in ['float64']:
        print("**Estatísticas**:")
        print(f"  - Média: {cito_mamo[column].mean():,.2f}")
        print(f"  - Mediana: {cito_mamo[column].median():,.2f}")
        print(f"  - Mínimo: {cito_mamo[column].min():,}")
        print(f"  - Máximo: {cito_mamo[column].max():,}")

        q1 = cito_mamo[column].quantile(0.25)
        q3 = cito_mamo[column].quantile(0.75)
        iqr = q3 - q1
        std_robust = iqr / 1.349  # Fórmula robusta: σ ≈ IQR / 1.349 (para distribuição normal)
        print(f"  - Desvio padrão: {std_robust:,.2f}")
        print(f"  - IQR (Intervalo Interquartílico): {iqr:,.2f}")
    else:
        unique_values = cito_mamo[column].dropna().unique()
        if len(unique_values) <= 10:
            print(f"**Valores únicos**: {', '.join(str(v) for v in unique_values)}")
        else:
            print(f"**Primeiros valores únicos**: {', '.join(str(v) for v in unique_values[:10])}...")

    print()

print("---")
print("*Documentação gerada automaticamente em", pd.Timestamp.now().strftime("%d/%m/%Y %H:%M"))

# DOCUMENTAÇÃO DO DATASET - EXAMES CITOPATOLÓGICOS

## Visão Geral
- **Total de registros**: 114
- **Total de colunas**: 6
- **Última atualização**: 24 de novembro de 2025

## Descrição Detalhada das Colunas

### 1. estado
**Tipo de dado**: str
**Valores únicos**: 1
**Valores nulos**: 0 (0.0%)
**Valores únicos**: CE

### 2. municipio
**Tipo de dado**: str
**Valores únicos**: 17
**Valores nulos**: 0 (0.0%)
**Primeiros valores únicos**: Brejo Santo, Crato, Juazeiro do Norte, Barbalha, Farias Brito, Missão Velha, Campos Sales, Várzea Alegre, Abaiara, Mauriti...

### 3. valor_indicador
**Tipo de dado**: float64
**Valores únicos**: 108
**Valores nulos**: 0 (0.0%)
**Estatísticas**:
  - Média: 3,345.48
  - Mediana: 1,255.00
  - Mínimo: 1.0
  - Máximo: 22,309.0
  - Desvio padrão: 2,825.24
  - IQR (Intervalo Interquartílico): 3,811.25

### 4. ano_mes
**Tipo de dado**: int64
**Valores únicos**: 10
**Valores nulos**: 0 (0.0%)
**Valores únicos**: 201612, 201712, 201812, 201912, 202012, 202112, 202

#### _Desvio padrão e seus detalhamentos_

Nesse ponto foi utilizado a métrica IQR, para facilitar a captura de outliers

MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO <br>
O que é:

> O IQR divide os dados em 4 partes iguais (quartis). <br>
> Valores que saem dos limites IQR são considerados outliers.

In [53]:
print("MÉTRICAS DE PADRÃO E DETECÇÃO DE ANOMALIAS")

if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador']
    
    print(f"\n MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO")
    print("-" * 100)
    
    q1 = valor_indicador.quantile(0.25)
    q2 = valor_indicador.quantile(0.50)
    q3 = valor_indicador.quantile(0.75)
    iqr = q3 - q1
    
    print(f" VALORES DO DATASET:")
    print(f"   • Q1 (25º percentil): {q1:,.2f}")
    print(f"   • Q2 (50º percentil/Mediana): {q2:,.2f}")
    print(f"   • Q3 (75º percentil): {q3:,.2f}")
    print(f"   • IQR (Q3 - Q1): {iqr:,.2f}")
    
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    
    print(f"\n QUANDO FOGE DO PADRÃO (método IQR - 1.5x):")
    print(f"   • Limite inferior: Q1 - 1.5 × IQR = {q1:,.2f} - 1.5 × {iqr:,.2f} = {limite_inferior:,.2f}")
    print(f"   • Limite superior: Q3 + 1.5 × IQR = {q3:,.2f} + 1.5 × {iqr:,.2f} = {limite_superior:,.2f}")
    print(f"   • Outliers: valores < {limite_inferior:,.2f} ou > {limite_superior:,.2f}")
    
    outliers_iqr = valor_indicador[(valor_indicador < limite_inferior) | (valor_indicador > limite_superior)]
    
    print(f"\n NO DATASET:")
    print(f"   • Total de outliers (IQR): {len(outliers_iqr)} ({len(outliers_iqr)/len(valor_indicador)*100:.1f}%)")
    print(f"   • Valores mínimo/máximo reais: {valor_indicador.min():,.2f} / {valor_indicador.max():,.2f}")
    
    print(f"   • Valores fora do intervalo [{limite_inferior:,.2f}, {limite_superior:,.2f}] são anomalias")

else:
    print(" Coluna 'valor_indicador' não encontrada no dataset!")

MÉTRICAS DE PADRÃO E DETECÇÃO DE ANOMALIAS

 MÉTRICA: IQR - INTERVALO INTERQUARTÍLICO
----------------------------------------------------------------------------------------------------
 VALORES DO DATASET:
   • Q1 (25º percentil): 155.00
   • Q2 (50º percentil/Mediana): 1,255.00
   • Q3 (75º percentil): 3,966.25
   • IQR (Q3 - Q1): 3,811.25

 QUANDO FOGE DO PADRÃO (método IQR - 1.5x):
   • Limite inferior: Q1 - 1.5 × IQR = 155.00 - 1.5 × 3,811.25 = -5,561.88
   • Limite superior: Q3 + 1.5 × IQR = 3,966.25 + 1.5 × 3,811.25 = 9,683.12
   • Outliers: valores < -5,561.88 ou > 9,683.12

 NO DATASET:
   • Total de outliers (IQR): 13 (11.4%)
   • Valores mínimo/máximo reais: 1.00 / 22,309.00
   • Valores fora do intervalo [-5,561.88, 9,683.12] são anomalias


Mostrando quem faz parte desse desvio...

In [54]:
print("ANOMALIAS DETECTADAS: Linhas que Fogem do Padrão (Outliers via IQR)")

if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador']

    q1 = valor_indicador.quantile(0.25)
    q3 = valor_indicador.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers_df = cito_mamo[(valor_indicador < limite_inferior) | (valor_indicador > limite_superior)].copy()

    outliers_df['tipo_outlier'] = outliers_df['valor_indicador'].apply(
        lambda x: 'Abaixo da Métrica' if x < limite_inferior else 'Acima da Métrica'
    )

    mediana = valor_indicador.median()
    outliers_df['desvio_mediana'] = outliers_df['valor_indicador'] - mediana

    print(f"\n RESUMO DOS OUTLIERS:")
    print(f"-" * 50)
    print(f"Total de registros no dataset: {len(cito_mamo)}")
    print(f"Total de outliers detectados: {len(outliers_df)} ({len(outliers_df)/len(cito_mamo)*100:.1f}%)")
    print(f"Limites IQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")
    print(f"Mediana do dataset: {mediana:,.2f}\n")

    if len(outliers_df) > 0:
        print(f"TABELA COMPLETA DOS OUTLIERS:")
        print(f"-" * 120)

        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        cols_importantes = ['valor_indicador', 'tipo_outlier', 'desvio_mediana']
        outras_cols = [col for col in outliers_df.columns if col not in cols_importantes]
        ordem_cols = cols_importantes + outras_cols

        print(outliers_df[ordem_cols].to_string(index=True))

        print(f"\n" + "-" * 120)
        print(f"INTERPRETAÇÃO:")
        print(f"- **Outlier Inferior**: Valores muito baixos (abaixo de {limite_inferior:,.2f})")
        print(f"- **Outlier Superior**: Valores muito altos (acima de {limite_superior:,.2f})")
        print(f"- **Desvio da Mediana**: Quanto o valor se afasta da mediana ({mediana:,.2f})")
        print(f"- Valores positivos = acima da mediana | Valores negativos = abaixo da mediana")
        print(f"- Quanto maior o desvio absoluto, mais extrema é a anomalia\n")

        print(f"-" * 50)
        print(f"Valor indicador - Mínimo: {outliers_df['valor_indicador'].min():,.2f}")
        print(f"Valor indicador - Máximo: {outliers_df['valor_indicador'].max():,.2f}")
        print(f"Valor indicador - Média: {outliers_df['valor_indicador'].mean():,.2f}")
        print(f"Desvio da mediana - Médio: {outliers_df['desvio_mediana'].mean():,.2f}")
        print(f"Outliers inferiores: {len(outliers_df[outliers_df['tipo_outlier'] == 'Outlier Inferior'])}")
        print(f"Outliers superiores: {len(outliers_df[outliers_df['tipo_outlier'] == 'Outlier Superior'])}")

    else:
        print("Todos os valores estão dentro dos limites considerados normais pelo método IQR.")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")

ANOMALIAS DETECTADAS: Linhas que Fogem do Padrão (Outliers via IQR)

 RESUMO DOS OUTLIERS:
--------------------------------------------------
Total de registros no dataset: 114
Total de outliers detectados: 13 (11.4%)
Limites IQR: [-5,561.88, 9,683.12]
Mediana do dataset: 1,255.00

TABELA COMPLETA DOS OUTLIERS:
------------------------------------------------------------------------------------------------------------------------
    valor_indicador      tipo_outlier  desvio_mediana estado          municipio  ano_mes     data_atualizacao      tipo_exame
1           16742.0  Acima da Métrica         15487.0     CE              Crato   201612  2026-03-17 01:25:03  Citopatologico
5           19052.0  Acima da Métrica         17797.0     CE              Crato   201712  2026-03-17 01:22:16  Citopatologico
9           13976.0  Acima da Métrica         12721.0     CE              Crato   201812  2026-03-17 01:20:07  Citopatologico
10           9735.0  Acima da Métrica          8480.0     CE  

### **Análise Exploratória (EDA- Exploratory Data Analysis)**

##### _Medidas de posições_

> - Média: Valor central aritmético <br>
> - Mediana: Valor central quando ordenado <br>
> - Moda: Valor(es) mais frequente(s)

In [75]:
if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador'].dropna()  # Remover NaN para cálculos

    print("MEDIDAS DE POSIÇÃO")
    print("-" * 50)

    media = valor_indicador.mean()
    mediana = valor_indicador.median()
    moda = valor_indicador.mode()

    print(f"Média: {media:,.2f}")
    print(f"Mediana: {mediana:,.2f}")
    if len(moda) > 0:
        if len(moda) == 1:
            print(f"Moda: {moda.iloc[0]:,.2f}")
        else:
            print(f"Modas ({len(moda)}): {', '.join([f'{m:,.2f}' for m in moda])}")
    else:
        print("Moda: Não existe (distribuição uniforme)")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")

MEDIDAS DE POSIÇÃO
--------------------------------------------------
Média: 3,345.48
Mediana: 1,255.00
Moda: 1.00


##### _Separatrizes (Quartis e Percentis)_

> - Quartis: Q1 (25%), Q2 (50% - mediana), Q3 (75%) <br>
> - Percentis: P10, P90, P95

In [74]:
if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador'].dropna()  # Remover NaN para cálculos

    print(f"\nSEPARATRIZES")
    print("-" * 50)

    q1 = valor_indicador.quantile(0.25)
    q2 = valor_indicador.quantile(0.50)  # Mesmo que mediana
    q3 = valor_indicador.quantile(0.75)

    print(f"Quartis:")
    print(f"      • Q1 (25º percentil): {q1:,.2f}")
    print(f"      • Q2 (50º percentil/Mediana): {q2:,.2f}")
    print(f"      • Q3 (75º percentil): {q3:,.2f}")

    # Percentis adicionais
    p10 = valor_indicador.quantile(0.10)
    p90 = valor_indicador.quantile(0.90)
    p95 = valor_indicador.quantile(0.95)

    print(f"\nOutros Percentis:")
    print(f"      • P10 (10º percentil): {p10:,.2f}")
    print(f"      • P90 (90º percentil): {p90:,.2f}")
    print(f"      • P95 (95º percentil): {p95:,.2f}")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")


SEPARATRIZES
--------------------------------------------------
Quartis:
      • Q1 (25º percentil): 155.00
      • Q2 (50º percentil/Mediana): 1,255.00
      • Q3 (75º percentil): 3,966.25

Outros Percentis:
      • P10 (10º percentil): 64.60
      • P90 (90º percentil): 10,595.30
      • P95 (95º percentil): 14,944.10


##### _Medidas de dispersão_

> - Amplitude: Diferença entre máximo e mínimo <br>
> - Variância: Média dos quadrados dos desvios <br>
> - Desvio Padrão: Raiz quadrada da variância <br>
> - IQR: Intervalo interquartílico (Q3 - Q1) <br>
> - Coeficiente de Variação: Desvio padrão / média × 100% <br>

In [77]:
if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador'].dropna()  # Remover NaN para cálculos

    print(f"\nMEDIDAS DE DISPERSÃO")
    print("-" * 50)

    amplitude = valor_indicador.max() - valor_indicador.min()

    variancia = valor_indicador.var()

    desvio_padrao = valor_indicador.std()

    iqr = q3 - q1

    cv = (desvio_padrao / media) * 100

    print(f"Amplitude: {amplitude:,.2f}")
    print(f"Variância: {variancia:,.2f}")
    print(f"Desvio Padrão: {desvio_padrao:,.2f}")
    print(f"IQR (Intervalo Interquartílico): {iqr:,.2f}")
    print(f"Coeficiente de Variação: {cv:.2f}%")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")


MEDIDAS DE DISPERSÃO
--------------------------------------------------
Amplitude: 22,308.00
Variância: 25,212,679.79
Desvio Padrão: 5,021.22
IQR (Intervalo Interquartílico): 3,811.25
Coeficiente de Variação: 150.09%


##### _Assimetria (SKEWNESS)_

> - Coeficiente: Mede o grau de assimetria <br>
> - Interpretação: Simétrica, assimétrica à direita/esquerda <br>
> - Classificação: Leve, moderada ou forte

In [78]:
if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador'].dropna()  # Remover NaN para cálculos

    print(f"\nASSIMETRIA (SKEWNESS)")
    print("-" * 50)

    skewness = valor_indicador.skew()

    print(f"Coeficiente de Assimetria: {skewness:.4f}")

    if skewness > 0:
        print("Interpretação: Distribuição assimétrica à DIREITA (cauda alongada à direita)")
        print("Significado: Média > Mediana, valores extremos altos mais frequentes")
    elif skewness < 0:
        print("Interpretação: Distribuição assimétrica à ESQUERDA (cauda alongada à esquerda)")
        print("Significado: Média < Mediana, valores extremos baixos mais frequentes")
    else:
        print("Interpretação: Distribuição SIMÉTRICA (aproximadamente normal)")

    if abs(skewness) < 0.5:
        print("Classificação: Assimetria LEVE")
    elif abs(skewness) < 1.0:
        print("Classificação: Assimetria MODERADA")
    else:
        print("Classificação: Assimetria FORTE")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")


ASSIMETRIA (SKEWNESS)
--------------------------------------------------
Coeficiente de Assimetria: 2.1405
Interpretação: Distribuição assimétrica à DIREITA (cauda alongada à direita)
Significado: Média > Mediana, valores extremos altos mais frequentes
Classificação: Assimetria FORTE


##### _Curtose (KURTOSIS)_

> - Coeficiente: Mede o "achatamento" da distribuição <br>
> - Interpretação: Leptocúrtica (pontuda), platicúrtica (achatada), mesocúrtica (normal) <br>
> - Classificação: Alta, baixa ou normal

In [79]:
if 'valor_indicador' in cito_mamo.columns:
    valor_indicador = cito_mamo['valor_indicador'].dropna()  # Remover NaN para cálculos

    print(f"\nCURTOSE (KURTOSIS)")
    print("-" * 50)

    kurtosis = valor_indicador.kurtosis()

    print(f"Coeficiente de Curtose: {kurtosis:.4f}")

    if kurtosis > 0:
        print("Interpretação: Distribuição LEPTOCÚRTICA (mais pontuda, caudas pesadas)")
        print("Significado: Valores extremos mais frequentes que na distribuição normal")
    elif kurtosis < 0:
        print("Interpretação: Distribuição PLATICÚRTICA (mais achatada, caudas leves)")
        print("Significado: Valores extremos menos frequentes que na distribuição normal")
    else:
        print("Interpretação: Distribuição MESOCÚRTICA (semelhante à normal)")

    if kurtosis > 0.5:
        print("Classificação: Curtose ALTA (muitos outliers esperados)")
    elif kurtosis < -0.5:
        print("Classificação: Curtose BAIXA (poucos outliers)")
    else:
        print("Classificação: Curtose NORMAL")

else:
    print("Coluna 'valor_indicador' não encontrada no dataset!")


CURTOSE (KURTOSIS)
--------------------------------------------------
Coeficiente de Curtose: 4.1101
Interpretação: Distribuição LEPTOCÚRTICA (mais pontuda, caudas pesadas)
Significado: Valores extremos mais frequentes que na distribuição normal
Classificação: Curtose ALTA (muitos outliers esperados)


##### **_RESUMO GERAL DA DISTRIBUIÇÃO:_**

- Total de observações: 114
- Valores únicos: 108
- Valores nulos: 0

**_CLASSIFICAÇÃO GERAL:_**
- DISTRIBUIÇÃO NÃO NORMAL
- Tendência: Valores altos extremos
- Característica: Muitos outliers esperados